# Notebook 12 — Nested regularization validation

**Goal:** improve the original 3-class next-bar classifier without optimizing on the final holdout. Five expanding outer folds, an inner chronological validation split to choose Logistic L2 `C` by log-loss, and comparison with fixed `C=0.1` and the prior-class baseline. This notebook **does not evaluate** the latest 20% holdout.

Logistic L1 remains a previous reference; the first controlled experiment tunes the more efficient L2 model. ROC curves are macro one-vs-rest, and model fit times are recorded.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks': ROOT=ROOT.parent
if not (ROOT/'src'/'stock_ml_dynamic').exists():
    raise RuntimeError('Open the notebook from the stock_ml_dynamic repository')
sys.path.insert(0,str(ROOT/'src'))
from stock_ml_dynamic.pooled_models import load_master
from stock_ml_dynamic.regularization_diagnostics import (run_regularization_diagnostics, summary_regularization, plot_regularization)

candidates=[ROOT/'artifacts'/'master_tables'/'1d',ROOT/'notebooks'/'artifacts'/'master_tables'/'1d']
required=('master_labeled.parquet','latest_inference.parquet','transformers.joblib','manifest.json')
master_dir=next((d for d in candidates if all((d/f).exists() for f in required)),None)
if master_dir is None: raise FileNotFoundError(f'Missing master table in {candidates}. Run Notebook 03 first.')
master,latest,bundle,manifest=load_master(master_dir)
print('Master:',master_dir,'rows:',len(master),'symbols:',master.symbol.nunique())

In [ ]:
# The 20% final holdout remains untouched. Only inner validation selects C.
N_FOLDS=5
C_GRID=(0.03,0.1,0.3,1.0)
MAX_TRAIN_ROWS=120_000
OUTPUT=ROOT/'artifacts'/'regularization_diagnostics'/'1d'
scores,tuning,roc,eligibility,development_end=run_regularization_diagnostics(
    master,n_folds=N_FOLDS,c_grid=C_GRID,max_train_rows=MAX_TRAIN_ROWS,output_dir=OUTPUT)
print('Development ends:',development_end)
print('Outer results:',len(scores),'inner trials:',len(tuning))

In [ ]:
display(summary_regularization(scores))
display(tuning.pivot(index='fold',columns='C',values='log_loss'))
display(scores[['fold','model','selected_C','accuracy','balanced_accuracy','roc_auc','log_loss','fit_seconds']])

In [ ]:
plot_regularization(scores,tuning,roc,OUTPUT)
print('Saved to:',OUTPUT)
print('Final holdout status: NOT EVALUATED')

## Interpretation and next decision

Compare **tuned L2 vs fixed L2 on the same outer folds**. An improvement in mean accuracy without comparable ROC-AUC/log-loss improvement may be noise. Check variation across folds and whether selected `C` values are stable. The newest 20% remains reserved for a single final evaluation after freezing target, features, regularization, and model-selection rules.

**Important:** Outer CV is used for reporting generalization estimates, not for picking additional hyperparameters after seeing the results. Repeatedly redesigning experiments around these folds can still overfit development validation.